In [ ]:

from tqdm import tqdm
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from scipy.spatial.distance import pdist

In [2]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [6]:
from data_utils import get_images, get_labels
from feature_utils import get_sobel_features, get_gabor_features, generate_gabor_kernel, get_local_binary_pattern

In [ ]:
disaster_list = ["socal-fire", "midwest-flooding"]

In [ ]:
data = {}
split = "train"
with open('config.json') as config_file:
    config = json.load(config_file)
    data_dir = config['data_dir']

for disaster in disaster_list:
    print(f"Loading {split} images and labels for {disaster} dataset...")
    images = get_images(data_dir, disaster, split=split)
    labels = get_labels(data_dir, disaster, split=split)
    data[disaster] = {"images": images, "labels": labels}

Loading test images and labels for flooding-fire dataset...


FileNotFoundError: [Errno 2] No such file or directory: '../satellite-image-data/flooding-fire/test_images.npz'

In [9]:
def get_gabor_aggregate_features(image, kernel):
    gabor_filtered_image = get_gabor_features(image, kernel)
    var = np.var(gabor_filtered_image)
    mean = np.mean(gabor_filtered_image)
    return mean, var

def get_encoded_gabor_features(image, kernels) -> np.ndarray:
    feats = np.empty((len(kernels), 2), dtype=float)
    for i, kernel in enumerate(kernels):
        feats[i, 0], feats[i, 1] = get_gabor_aggregate_features(image, kernel)
    return feats

In [11]:
def crop_image(img):
    crop = np.argwhere(np.sum(img, axis=2) != 0)
    x_min, x_max = np.min(crop[:, 0]), np.max(crop[:, 0])
    y_min, y_max = np.min(crop[:, 1]), np.max(crop[:, 1])
    return img[x_min:x_max, y_min:y_max]

In [10]:
kernels = []
for theta in range(4):
    theta = theta / 4.0 * np.pi
    for sigma in (1, 5):
        for frequency in (0.05, 0.25):
            kernel = generate_gabor_kernel(theta=theta, sigma=sigma, frequency=frequency)
            kernels.append(kernel)

In [15]:
gabor_filter_feature_list = []
gabor_filter_label_list = []
for disaster, disaster_data in data.items():
    parallel = Parallel(n_jobs=7, return_as="generator")
    for features, disaster in tqdm(parallel(delayed(get_features)(image, disaster) for image in disaster_data['images'])):
        gabor_filter_feature_list.append(features)
        gabor_filter_label_list.append(disaster)
    
y, X = np.array(gabor_filter_label_list), np.vstack(gabor_filter_feature_list)

135it [03:15,  1.45s/it]


KeyboardInterrupt: 